Read new/changed records based on the `updated_on` watermark and writes them append-only into `chicago_crime.bronze.crimes`.

# SECTION 0: Imports

In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, TimestampNTZType, TimestampType

In [0]:
%run ../config

# SECTION 1: Derived table names + Widgets

In [0]:
TARGET_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.crimes"
CONTROL_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.load_control"

CONTROL_SCHEMA = StructType([
    StructField("last_watermark", TimestampNTZType(), True),
    StructField("last_update", TimestampType(), True),
])

DATASET_ID = 'ijzp-q8t2'
ENDPOINT = f"https://{SOCRATA_DOMAIN}/api/v3/views/{DATASET_ID}/query.json"
PAGE_SIZE = 50000
CHUNK_DAYS = 30
MAX_PAGES_PER_CHUNK = 500

DEFAULT_START_WATERMARK = "2015-01-01T00:00:00.000"

CRIME_COLUMNS = [
    "id", "case_number", "date", "block", "iucr", "primary_type",
    "description", "location_description", "arrest", "domestic",
    "beat", "district", "ward", "community_area", "fbi_code",
    "x_coordinate", "y_coordinate", "year", "updated_on",
    "latitude", "longitude", "location",
]

In [0]:
# Widgets let this notebook be parameterized from outside 
# (eg. ADF, or a manual run for ad hoc reprocessing) without editing any code

dbutils.widgets.dropdown(
    name='run_mode', 
    defaultValue='incremental',
    choices=['incremental', 'manual_backfill'],
    label='run_mode'
)
dbutils.widgets.text(
    name='manual_start_date',
    defaultValue='',
    label='Manual: start date (YYYY-MM-DDTHH:mm:ss.sss)'
)
dbutils.widgets.text(
    name='manual_end_date',
    defaultValue='',
    label='Manual: end date (YYYY-MM-DDTHH:mm:ss.sss) (Empty = Infinite)'
)

In [0]:
run_mode = dbutils.widgets.get("run_mode")
manual_start_date = dbutils.widgets.get("manual_start_date").strip()
manual_end_date = dbutils.widgets.get("manual_end_date").strip()

# SECTION 2: Get the API token from the secret scope

In [0]:
app_token = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

# SECTION 3: Resovle the starting watermark

In [0]:
def get_last_watermark() -> str:

    if not spark.catalog.tableExists(CONTROL_TABLE):
        print(f"[CONTROL TABLE] {CONTROL_TABLE} does not exist yet - first run, using DEFAULT_START_WATERMARK")
        return DEFAULT_START_WATERMARK
    control_df = spark.table(CONTROL_TABLE)

    if control_df.count() == 0:
        print(f"[CONTROL TABLE] {CONTROL_TABLE} exists but is empty - using DEFAULT_START_WATERMARK")
        return DEFAULT_START_WATERMARK
    
    max_watermark = control_df.selectExpr("max(last_watermark) as max_watermark").collect()[0]["max_watermark"]

    return max_watermark.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]

In [0]:
if run_mode == "manual_backfill":
    if not manual_start_date:
        raise ValueError("run_mode=manual_backfill requires manual_start_date to be set")

    # Guard: manual_backfill must never be the very first run - it can only reprocess a window that a prior incremental run already covered.

    control_table_is_empty = (
        not spark.catalog.tableExists(CONTROL_TABLE)
        or spark.table(CONTROL_TABLE).count() == 0
    )

    if control_table_is_empty:
        raise ValueError(
            "manual_backfill cannot be used before the first incremental run."
            "Run with run_mode=incremental first to seed the control table"
            "(it will backfill from DEFAULT_START_WATERMARK automatically), "
            "then use manual_backfill only to reprocess a specific window later."
        )
    
    last_watermark = manual_start_date
    print(f"[MANUAL BACKFILL] fetching from {last_watermark} (control table will NOT be touched)")
else:
    last_watermark = get_last_watermark()
    print(f"[INCREMENTAL] fetching updated_on > {last_watermark}")

In [0]:
end_watermark = manual_end_date if (run_mode == 'manual_backfill' and manual_end_date) else None

# SECTION 4: SoQL + Fetch Helpers

In [0]:
def build_soql(start_watermark, end_watermark=None):
    where_clause = f"updated_on > '{start_watermark}'"
    if end_watermark:
        where_clause += f" AND updated_on <= '{end_watermark}'"
    return f"SELECT * WHERE {where_clause} ORDER BY updated_on ASC, id ASC"

In [0]:
def fetch_page(soql, page_number, page_size):
    headers = {"X-App-Token": app_token}
    payload = {
        "query": soql,
        "page": {"pageNumber": page_number, "pageSize": page_size},
        "includeSynthetic": False,
    }
    response = requests.post(ENDPOINT, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()

In [0]:
def fetch_chunk(chunk_start, chunk_end, page_size=PAGE_SIZE):
    """
    Fetch every page for ONE date-range chunk and return it as a single list.
    """

    soql = build_soql(chunk_start, chunk_end)
    chunk_rows = []
    page_number = 1
    
    while True:
        print(f"[FETCH] chunk {chunk_start} -> {chunk_end}, page {page_number}")
        page_rows = fetch_page(soql, page_number, page_size)
        if not page_rows:
            break
        chunk_rows.extend(page_rows)
        if len(page_rows) < page_size:
            break
        page_number += 1
        if page_number > MAX_PAGES_PER_CHUNK:
            raise RuntimeError(
                f"Chunk {chunk_start} -> {chunk_end} exceeded {MAX_PAGES_PER_CHUNK} "
                f"pages ({MAX_PAGES_PER_CHUNK * page_size:,} rows) without finishing. "
                f"This is far beyond even a legitimate bulk-update spike and strongly "
                f"suggests a pagination bug rather than real data. Stopping to avoid "
                f"an unbounded fetch."
            )
    return chunk_rows

In [0]:
def generate_date_chunks(start_str, end_str, chunk_days):
    """
    Splits [start, end] into smaller (start, end] windows so each SODA3 query only needs to scan a limited slice of data. This sidesteps the deep page number slowdown, because within one chunk the row count small enough that pageNumber rarely needs to go past 1-2 page.
    """

    start_dt = datetime.strptime(start_str, "%Y-%m-%dT%H:%M:%S.%f")
    end_dt = datetime.strptime(end_str, "%Y-%m-%dT%H:%M:%S.%f")
    chunks = []
    current = start_dt
    
    while current < end_dt:
        chunk_end = min(current + timedelta(days=chunk_days), end_dt)
        chunks.append((
            current.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3],
            chunk_end.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
        ))
        current = chunk_end
    return chunks

# SECTION 5: Fetch, convert, write, and checkpoint the watermark - one chunk at a time

In [0]:
effective_end = end_watermark or datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
chunks = generate_date_chunks(last_watermark, effective_end, CHUNK_DAYS)

total_rows_written = 0
for chunk_start, chunk_end in chunks:
    chunk_rows = fetch_chunk(chunk_start, chunk_end)

    if not chunk_rows:
        print(f"[INFO] chunk {chunk_start} -> {chunk_end}: no rows, skipping write")
        continue

    string_rows = [
        Row(**{col: (str(row[col]) if col in row else None) for col in CRIME_COLUMNS})
        for row in chunk_rows
    ]
    chunk_df = spark.createDataFrame(string_rows)
    chunk_df = chunk_df.withColumn("_ingested_at", F.current_timestamp())

    (
        chunk_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(TARGET_TABLE)
    )
    
    rows_in_chunk = chunk_df.count()
    total_rows_written += rows_in_chunk
    print(f"[WRITE] chunk {chunk_start} -> {chunk_end}:appended {rows_in_chunk}")

    if run_mode == "incremental":
        chunk_watermark = chunk_df.selectExpr("max(updated_on) as max_updated_on").collect()[0]["max_updated_on"]
        chunk_watermark_dt = datetime.strptime(chunk_watermark, "%Y-%m-%dT%H:%M:%S.%f")

        control_row = spark.createDataFrame(
            [Row(last_watermark=chunk_watermark_dt, updated_at=datetime.utcnow())],
            schema=CONTROL_SCHEMA
        )
        control_row.write.format("delta").mode("append").saveAsTable(CONTROL_TABLE)
        print(f"[CONTROL TABLE] checkpoint advanced to {chunk_watermark_dt}")

    del chunk_rows, string_rows, chunk_df   

print(f"[DONE] total rows written across all chunks: {total_rows_written}")

# SECTION 6: Final status (the control table was already checkopinted per chunk in SECTION 5)

In [0]:
if run_mode == "manual_backfill":
    print("[CONTROL TABLE] manual_backfill run - control table intentionally left untouched")
elif total_rows_written == 0:
    print("[CONTROL TABLE] no new rows this run - watermark unchanged")
else:
    print("[CONTROL TABLE] watermark checkpointed incrementally per chunk during this run - see SECTION 5 logs above")